## Context for this notebook

I ran a ShinkaEvolve experiment on the Komlos conjecture for n=12 a couple of months ago, and re-discovered the construction of Kunisky (2021), which is the best known lower bound.

I was using the following LLM models:
- GPT 5 Nano
- Gemini 3 Pro
- Gemini 3 Flash
- GPT Codex 5.2
- o4 mini

The whole experiment cost around $10 and ran in a couple of hours, so it might be a little too large to replicate on a one-shot.

# Results of ShinkaEvolve on Komlos lower bounds for n=12

In [16]:
# EVOLVE-BLOCK-START
import numpy as np
import scipy.linalg as sp

# Marco: I modified the function signature to make it easier to change num_trials and iterations, Shinka only evolved construct_vectors(n)

def construct_vectors(n: int = 12, num_trials: int = 8, iterations: int = 600):  
    """
    Hybrid Bottleneck Optimizer.
    Combines aggressive pattern masking, rank-weighted SoftMin, and manifold optimization
    to maximize the minimum discrepancy.
    """
    # ---------------------------------------------------------
    # 1. Precompute Sign Patterns
    # ---------------------------------------------------------
    if n <= 16:
        ns = 1 << n
        idxs = np.arange(ns, dtype=np.uint32)
        bits = (idxs[:, None] >> np.arange(n)[None, :]) & 1
        signs_matrix = (2.0 * bits - 1.0).T.astype(np.float32)
    else:
        signs_matrix = np.random.choice([-1, 1], size=(n, 8192)).astype(np.float32)

    def get_score(A):
        vals = np.max(np.abs(A @ signs_matrix), axis=0)
        return np.min(vals)

    def normalize_cols(A):
        return A / np.linalg.norm(A, axis=0)

    best_global_A = None
    best_global_score = -np.inf

    # ---------------------------------------------------------
    # 2. Optimization Loop Over Trials
    # ---------------------------------------------------------
    for trial in range(num_trials):
        # Initialization Strategy
        if trial == 0:
            # Near-Identity
            A = normalize_cols(np.eye(n) + 0.1 * np.random.randn(n, n))
        elif trial == 1 and n % 4 == 0:
            # Near-Hadamard
            try:
                H = sp.hadamard(n).astype(np.float32)
                A = normalize_cols(H + 0.1 * np.random.randn(n, n))
            except:
                A = normalize_cols(np.random.randn(n, n))
        elif trial == 2:
            # Random Orthogonal
            Q, _ = np.linalg.qr(np.random.randn(n, n))
            A = Q.astype(np.float32)
        elif trial == 3:
            # DCT-like structured seed
            idx = np.arange(n)
            dct = np.cos(np.pi / n * (idx[:, None] + 0.5) * idx[None, :]).astype(np.float32)
            A = normalize_cols(dct + 0.06 * np.random.randn(n, n))
        elif best_global_A is not None and trial < num_trials - 2:
            # Evolutionary: Perturb best so far
            A = normalize_cols(best_global_A + 0.05 * np.random.randn(n, n))
        else:
            # Random Normal
            A = normalize_cols(np.random.randn(n, n))

        # Optimizer State (Adam)
        m = np.zeros_like(A)
        v = np.zeros_like(A)
        s_row = np.ones(n, dtype=np.float32) # Row-wise second moment

        # Persistent pattern boosting
        pattern_boost = np.ones(signs_matrix.shape[1], dtype=np.float32)

        # Hyperparameters
        beta1, beta2 = 0.88, 0.98
        beta2_row = 0.95
        eps = 1e-8
        lr_base = 0.04

        trial_best_A = A.copy()
        trial_best_score = get_score(A)
        last_improve = 0
        last_score = trial_best_score

        for it in range(1, iterations + 1):
            progress = it / iterations

            # -----------------------------------------------------
            # Forward Pass with Smooth Approximations
            # -----------------------------------------------------
            Y = A @ signs_matrix
            Y_abs = np.abs(Y)

            # Smooth Max over rows (Gamma schedule)
            gamma = 8.0 + 90.0 * (progress ** 0.8)
            m_k = np.max(Y_abs, axis=0)
            exp_Y = np.exp(gamma * (Y_abs - m_k))
            sum_exp_Y = np.sum(exp_Y, axis=0)
            # V approximates ||Ax||_inf for each pattern x
            V = m_k + np.log(sum_exp_Y + 1e-12) / gamma

            # Smooth Min over patterns (Beta schedule)
            # We want to maximize min(V), so we weight patterns with small V
            beta = 10.0 + 100.0 * (progress ** 0.9)
            V_min = np.min(V)

            # -----------------------------------------------------
            # Pattern Attention: Masking + Persistent Boosting + SoftMin
            # -----------------------------------------------------
            # Focus on the bottom p% of patterns (Bottleneck Masking)
            # Starts at 45% and narrows down to 8%
            p_thresh = 0.45 - 0.37 * (progress ** 0.6)
            if p_thresh * len(V) < 8: p_thresh = 8.0 / len(V)

            v_limit = np.percentile(V, p_thresh * 100)
            is_bottleneck = (V <= v_limit)

            # Update persistent boosting
            # Boost bottleneck patterns, decay others slightly
            pattern_boost[is_bottleneck] *= 1.02
            pattern_boost[~is_bottleneck] *= 0.998
            pattern_boost = np.clip(pattern_boost, 0.5, 30.0)

            mask_V = is_bottleneck.astype(np.float32)
            exp_V = np.exp(-beta * (V - V_min)) * mask_V * pattern_boost
            weights = exp_V / (np.sum(exp_V) + 1e-12)

            # -----------------------------------------------------
            # Gradient Computation
            # -----------------------------------------------------
            # Derivative of LSE w.r.t Y_abs is SoftMax probabilities
            probs = exp_Y / (sum_exp_Y + 1e-12)

            # Adaptive Top-K filtering on rows (Coordinate Focus)
            # As optimization proceeds, only the peak coordinate matters
            k_row = max(1, int(n * (0.6 - 0.45 * progress)))
            if k_row < n:
                # Keep top k probabilities, zero others
                thresh = np.partition(probs, -k_row, axis=0)[-k_row, :]
                probs_mask = (probs >= thresh[None, :]).astype(np.float32)
                probs = probs * probs_mask
                probs /= (np.sum(probs, axis=0) + 1e-12)

            # Chain rule: dObjective/dA = (weights * dV/dY * sign(Y)) @ X.T
            grad_Y = (weights[None, :] * probs) * np.sign(Y)
            grad = grad_Y @ signs_matrix.T

            # -----------------------------------------------------
            # Regularization
            # -----------------------------------------------------
            # Orthogonality penalty (Decaying)
            # Helps exploration early on, then relaxes for max discrepancy
            ortho_lambda = 0.04 * (1.0 - progress)
            if ortho_lambda > 1e-5:
                AtA = A.T @ A
                grad_ortho = A @ (AtA - np.eye(n))
                grad -= ortho_lambda * grad_ortho

            # Project gradient to tangent space of oblique manifold (unit columns)
            col_dots = np.sum(A * grad, axis=0)
            grad_proj = grad - A * col_dots[None, :]

            # -----------------------------------------------------
            # Adam Update
            # -----------------------------------------------------
            m = beta1 * m + (1 - beta1) * grad_proj
            v = beta2 * v + (1 - beta2) * (grad_proj**2)

            # Row-wise adaptive damping
            # Helps if one row becomes scaling bottleneck
            row_norm2 = np.sum(grad_proj**2, axis=1)
            s_row = beta2_row * s_row + (1 - beta2_row) * row_norm2
            row_scale = 1.0 / (np.sqrt(s_row) + eps)

            m_hat = m / (1 - beta1**it)
            v_hat = v / (1 - beta2**it)

            # Learning Rate Schedule: Cosine Annealing
            lr = lr_base * 0.5 * (1.0 + np.cos(np.pi * progress))
            step = lr * (m_hat / (np.sqrt(v_hat) + eps)) * row_scale[:, None]

            # Lightweight line search every 20 iters
            if it % 20 == 0:
                A_try = normalize_cols(A + step)
                score_try = get_score(A_try)
                if score_try + 1e-6 < last_score:
                    step *= 0.5

            A += step

            # Progress-aware retraction on plateau
            if it - last_improve > 40 and it % 20 == 0:
                if progress < 0.7:
                    Q, _ = np.linalg.qr(A)
                    A = normalize_cols(Q)
                else:
                    U, _, Vt = np.linalg.svd(A, full_matrices=False)
                    A = U @ Vt

            A = normalize_cols(A)

            # -----------------------------------------------------
            # Tracking & Early Stopping
            # -----------------------------------------------------
            if it % 15 == 0:
                sc = get_score(A)
                last_score = sc
                if sc > trial_best_score:
                    trial_best_score = sc
                    trial_best_A = A.copy()
                    last_improve = it
                elif sc < trial_best_score * 0.96:
                    # Decay LR if performance drops significantly
                    lr_base *= 0.95

                # Pruning: if far behind global best late in trial
                if best_global_score > -np.inf and progress > 0.7:
                    if trial_best_score < best_global_score * 0.92:
                         break

        if trial_best_score > best_global_score:
            best_global_score = trial_best_score
            best_global_A = trial_best_A

    return best_global_A

# EVOLVE-BLOCK-END

In [17]:
import numpy as np
import scipy.linalg as sp
from itertools import product

def normalize(vectors):
    return vectors / sp.norm(vectors, axis=0)

def discrepancy(vectors):
    """
    Compute min_{x ∈ {-1,+1}^n} ||Ax||_∞ via brute force.
    Only feasible for small n (say n ≤ 20).
    """
    vectors = normalize(vectors)
    n = vectors.shape[1]
    min_val = np.inf
    best_x = None
    
    for signs in product([-1, 1], repeat=n):
        x = np.array(signs)
        val = np.abs(vectors @ x).max()
        if val < min_val:
            min_val = val
            best_x = x
    
    return min_val, best_x

def run_komlos(n:int = 12):
    vectors = construct_vectors(n)
    # Calculate the maximum discrepancy
    cost, signs = discrepancy(vectors)
    return vectors, cost, signs

In [69]:
K = 50
scores = [0]*K
mats = [0]*K
sgns = [0]*K
for i in range(K):
    A = construct_vectors()
    score, signs = discrepancy(A)
    mats[i] = A
    scores[i] = score
    sgns[i] = signs

In [70]:
scores

[np.float64(1.992247716230157),
 np.float64(1.977766780765514),
 np.float64(2.0129482759613433),
 np.float64(1.9891858308418695),
 np.float64(2.006337384971439),
 np.float64(2.0052924968185426),
 np.float64(1.9848149406249203),
 np.float64(1.9908027452198584),
 np.float64(1.9706437445847769),
 np.float64(2.0056610353313418),
 np.float64(1.935989935669156),
 np.float64(2.0053088957665013),
 np.float64(2.011903490327933),
 np.float64(2.0056884980179666),
 np.float64(2.0057178523242447),
 np.float64(2.0050662625711397),
 np.float64(1.9934316854553344),
 np.float64(2.0124812181726393),
 np.float64(1.9658276326445048),
 np.float64(1.9724327166965936),
 np.float64(2.0052760458421597),
 np.float64(1.9578602638369529),
 np.float64(1.9874949545559826),
 np.float64(1.986348491710564),
 np.float64(1.9651205251178137),
 np.float64(1.9910091490766197),
 np.float64(2.00267810035864),
 np.float64(1.9810947764678362),
 np.float64(2.0049717158859606),
 np.float64(1.9813766983457872),
 np.float64(1.9630

In [75]:
max(scores)

np.float64(2.0129482759613433)

---

Let's check the constructor for a smaller value of `n`

In [27]:
A = construct_vectors(8)

In [28]:
score, signs = discrepancy(A)
score

np.float64(1.8782093240404332)

In [35]:
K = 100
scores = [0]*K
mats = [0]*K
sgns = [0]*K
for i in range(K):
    A = construct_vectors(8)
    score, signs = discrepancy(A)
    mats[i] = A
    scores[i] = score
    sgns[i] = signs

In [37]:
sorted(scores)

[np.float64(1.8223484427424084),
 np.float64(1.8223750288059146),
 np.float64(1.8223781895308482),
 np.float64(1.8223783646426757),
 np.float64(1.8223786759639953),
 np.float64(1.8223866097051238),
 np.float64(1.8223938920158105),
 np.float64(1.8223978791946174),
 np.float64(1.8224048877397667),
 np.float64(1.8224057486142728),
 np.float64(1.8266603752396766),
 np.float64(1.828077980553632),
 np.float64(1.8282087521550858),
 np.float64(1.829316877759993),
 np.float64(1.8331953439733895),
 np.float64(1.8378182153512441),
 np.float64(1.8389021069147964),
 np.float64(1.8402783959536464),
 np.float64(1.8413878995334487),
 np.float64(1.8422501462488787),
 np.float64(1.8452524464810267),
 np.float64(1.8461577312970627),
 np.float64(1.8487486375822226),
 np.float64(1.8514083244954236),
 np.float64(1.852341396740063),
 np.float64(1.8546739323298653),
 np.float64(1.8560023763055526),
 np.float64(1.8567095327366432),
 np.float64(1.861389643940417),
 np.float64(1.8621787091358462),
 np.float64(1.

In [39]:
np.argmax(scores)

np.int64(67)

In [40]:
scores[67]

np.float64(1.9318038618664306)

In [42]:
sgns[67]

array([-1,  1,  1,  1, -1, -1,  1, -1])

In [41]:
np.savetxt("shinka_komlos_n8.txt", mats[67], fmt="%.18e")

---

Larger experiment, still for n=8

In [ ]:
n = 8
num_trials = 10
iterations = 1000

K = 200
scores = [0]*K
mats = [0]*K
sgns = [0]*K
for i in range(K):
    A = construct_vectors(n, num_trials, iterations)
    score, signs = discrepancy(A)
    mats[i] = A
    scores[i] = score
    sgns[i] = signs

In [44]:
sorted(scores)[-10:-1]

[np.float64(1.916870903242814),
 np.float64(1.9310351811237418),
 np.float64(1.9317820991653403),
 np.float64(1.9317829889998086),
 np.float64(1.9317830147096855),
 np.float64(1.9317839840908972),
 np.float64(1.9317922925249658),
 np.float64(1.9317934846140485),
 np.float64(1.9317973204959995)]

After this, I analyzed the best matrix achieving discrepancy roughly $1.9318$ (that I saved above) using Julia